In [42]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import cv2

# --- Load dataset ---
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "/kaggle/input/intel-image-classification/seg_train/seg_train",
    image_size=(224, 224),
    batch_size=32
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "/kaggle/input/intel-image-classification/seg_test/seg_test",
    image_size=(224, 224),
    batch_size=32
)

# --- Build model ---
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(224,224,3))
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)  # Important
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(128, activation='relu')(x)
outputs = tf.keras.layers.Dense(6, activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# --- Train ---
history = model.fit(train_ds, validation_data=val_ds, epochs=1)



Found 14034 files belonging to 6 classes.
Found 3000 files belonging to 6 classes.
439/439 ━━━━━━━━━━━━━━━━━━━━ 420s 946ms/step - accuracy: 0.8584 - loss: 0.3883 - val_accuracy: 0.9150 - val_loss: 0.2300


In [43]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name='out_relu'):
    # Access the MobileNetV2 backbone
    backbone = model.get_layer('mobilenetv2_1.00_224')  # this is the base model layer

    # Get the convolutional layer inside backbone
    conv_layer = backbone.get_layer(last_conv_layer_name)

    # Build grad model: input -> conv_layer output + model output
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[conv_layer.output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        class_idx = tf.argmax(predictions[0])
        loss = predictions[:, class_idx]

    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0,1,2))

    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0)
    if tf.reduce_max(heatmap) != 0:
        heatmap /= tf.reduce_max(heatmap)

    return heatmap.numpy(), class_idx.numpy()
